# HNL mu-pi selection (cafpybara)

The cafpybara port of `hnl/notebooks/hnl_mupi_selection.ipynb`. Same frames, cuts and plots, built on
`cafpybara.analyses.hnl_mupi`:

* `load_mc` / `load_mchnl` collapse the cafpyana `*_savepfp` frames (one row per PFParticle) to one row per slice,
  with the longest and second-longest track-like PFP as the `long1` / `long2` column groups, and apply the
  generator-specific BCFM flash-time fixes.
* Truth only colours the plots (`signal` column); every cut in `DEFAULT_CUTS` is reco-only, including the
  beam window on the calibrated BCFM flash time (`FLASH_TIME_WINDOW`).
* One figure per HNL mass: BNB stacked by true category at 1e20 POT with its MC-stat band, HNL as a step line at its
  expected rate for 1e20 POT and the simulated |U_μ4|² (MeVPrtl flux × ray × decay weights, `weights_mc`), times
  `ana.HNL_DISPLAY_SCALE[m]`. PDFs are written to `PLOT_DIR` as `<level>_<var>_m<mass>.pdf`.
* The per-track (`trk_*`) plots of the original notebook are not ported yet.

In [ ]:
import sys
from pathlib import Path

sys.path[:0] = ["/exp/sbnd/app/users/sdas1/cafpyana", "/exp/sbnd/app/users/sdas1"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import cafpybara.analyses.hnl_mupi as ana
from cafpybara.analyses.hnl_mupi import config

# v1 (area-normalised HNL, no flash-time window) is in selection_study_cafpybara/.
PLOT_DIR = Path("/exp/sbnd/app/users/sdas1/hnl/plots/selection_study_cafpybara")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
MASSES = (300, 365)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

## 1. Load (cafpyana preselection already applied)

In [ ]:
STREAM = True   # read the frames from dCache over xrootd (no /pnfs mount, e.g. on JupyterHub)
if STREAM:
    ana.ensure_token()
    BNB_FILE = ana.xrootd_url(config.BNB_FILE)
    HNL_FILES = {m: ana.xrootd_url(config.HNL_FILES[m]) for m in MASSES}
else:
    BNB_FILE, HNL_FILES = config.BNB_FILE, config.HNL_FILES
bnb0, POT_BNB, _ = ana.load_mc(BNB_FILE)
hnl0, POT_HNL, HNL_INFO = {}, {}, {}
for m in MASSES:
    hnl0[m], POT_HNL[m], HNL_INFO[m] = ana.load_mchnl(HNL_FILES[m])   # info = {simU, hnlM}
W_HNL = {m: config.TARGET_POT / POT_HNL[m] for m in MASSES}   # times weights_mc
W = config.TARGET_POT / POT_BNB          # weight per BNB slice
print(f"BNB: {len(bnb0)} slices, POT = {POT_BNB:.3e}, weight to {config.TARGET_POT:.0e} POT = {W:.3f}")
for m, h in hnl0.items():
    sig = h.signal == ana.signal_dict['hnl']
    print(f"m{m}: {len(h)} slices, {int(sig.sum())} HNL-matched, MeVPrtl POT = {POT_HNL[m]:.4g}, "
          f"|U|^2 = {HNL_INFO[m]['simU']:.1e}, expected matched slices @ {config.TARGET_POT:.0e} POT = "
          f"{h.weights_mc[sig].to_numpy().sum() * W_HNL[m]:.1f}")
print("fixes:", ana.applied_fixes(bnb0))

## 2. Cut scans on the preselected sample
HNL efficiency per mass (rate-weighted) against the fraction of BNB kept, by class.

In [ ]:
F0 = {"bnb": bnb0, **{f"m{m}": h for m, h in hnl0.items()}}
c = lambda *p: tuple(list(p) + [""] * (6 - len(p)))
SCANS = [
    ("BCFM score >",                 c("slc", "barycenterFM", "score"),  [0.001, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5], ">"),
    ("BCFM ΔZ [cm] <",               c("slc", "barycenterFM", "deltaZ"), [20, 50, 100, 200], "<"),
    ("opt0 score >",                 c("slc", "opt0", "score"),          [0, 50, 100, 200, 400, 800], ">"),
    ("CR long-track dir y >",        c("slc", "nuid", "crlongtrkdiry"),  [-0.9, -0.8, -0.5, -0.3, 0], ">"),
    ("Pandora ν score >",            c("slc", "nu_score"),               [0.5, 0.55, 0.6, 0.65, 0.7], ">"),
    ("Track-like PFPs per slice >=", c("n_trk"),                         [1, 2, 3], ">="),
    ("Calibrated BCFM flash time [ns] in [lo, hi)", c("slc", "barycenterFM", "flashTime_calib"),
     [(-250, 2250), (-250, 2500), (-250, 3000), (-250, 3500), (0, 3500)], "window"),
    ("Bunch phase [ns] in [lo, hi)", c("slc", "barycenterFM", "flashTime_calib_mod"),
     [(0, 18.936), (4, 16), (6, 16), (8, 16), (6, 12), (8, 14)], "window"),
]
for title, col, thr, op in SCANS:
    print(title)
    display(ana.scan_cut(F0, lambda d, col=col: d[col], thr, op))

## 3. Cut flow
BNB in slices at 1e20 POT per true category; HNL as matched slices and efficiency, plus the expected matched
slices at 1e20 POT and the rate-weighted efficiency.

In [ ]:
display(ana.cutflow(F0, ana.DEFAULT_CUTS, weights={"bnb": W, **{f"m{m}": W_HNL[m] for m in MASSES}}).round(3))
bnb = ana.select(bnb0, ana.DEFAULT_CUTS)
hnl = {m: ana.select(h, ana.DEFAULT_CUTS) for m, h in hnl0.items()}
hnl_sig = {m: h[h.signal == ana.signal_dict["hnl"]] for m, h in hnl.items()}

## 4. Plot helper

In [ ]:
def plot(name, col, xlabel, bins=None, logy=False, show_fom=False):
    """One PDF per mass. bins=None: 50 bins over the 0.5-99.5 percentiles of BNB + both HNL masses."""
    if bins is None:
        bins = ana.auto_bins([bnb, *hnl_sig.values()], col)
        if bins is None:
            print("no valid entries:", name); return
    for m in MASSES:
        k = ana.HNL_DISPLAY_SCALE[m]
        res = ana.plot_mc_hnl(bnb, hnl_sig[m], col, bins, scale_nu=W, scale_hnl=W_HNL[m] * k,
                              hnl_label=ana.hnl_label(m, HNL_INFO[m]["simU"], k), log_y=logy, show_fom=show_fom,
                              overflow=False, xlabel=xlabel, title=" ", counts=True,
                              ylabel=f"Slices / bin ({config.TARGET_POT:.0e} POT)",
                              legend_kwargs={"fontsize": 8},
                              savefig=PLOT_DIR / f"{name}_m{m}{'_fom' if show_fom else ''}.pdf")
        plt.show()
        plt.close(res[0])

## 5. Slice-level variables (after the cuts)

In [ ]:
P = 18.936  # bunch period [ns]
SLICE_VARS = [
    ("nu_score",            c("slc", "nu_score"),                       "Pandora ν score",           np.linspace(0.5, 1, 51),  True),
    ("bcfm_score",          c("slc", "barycenterFM", "score"),          "BCFM score",                np.linspace(0, 2, 81),    True),
    ("bcfm_score_zoom",     c("slc", "barycenterFM", "score"),          "BCFM score (zoom)",         np.linspace(0, 0.3, 61),  True),
    ("bcfm_deltaZ",         c("slc", "barycenterFM", "deltaZ"),         "BCFM ΔZ [cm]",              None,                     True),
    ("bcfm_chi2",           c("slc", "barycenterFM", "chi2"),           "BCFM χ²",                   None,                     True),
    ("bcfm_chargeTotal",    c("slc", "barycenterFM", "chargeTotal"),    "BCFM total charge",         None,                     True),
    ("bcfm_flashPEs",       c("slc", "barycenterFM", "flashPEs"),       "BCFM flash PEs",            None,                     True),
    ("bcfm_flashTime_corr", c("slc", "barycenterFM", "flashTime"),      "BCFM flash time, generator-corrected [µs]", None, False),
    ("bcfm_flashTime_phase", c("slc", "barycenterFM", "flashTime_calib_mod"), "(flash time - vertex ToF) mod 18.936 ns [ns]",
     np.linspace(0, P, 25), False),
    ("opt0_score",          c("slc", "opt0", "score"),                  "opt0 score",                None,                     True),
    ("opt0_time",           c("slc", "opt0", "time"),                   "opt0 time [µs]",            None,                     False),
    ("opt0_measPE",         c("slc", "opt0", "measPE"),                 "opt0 measured PE",          None,                     True),
    ("opt0_hypoPE",         c("slc", "opt0", "hypoPE"),                 "opt0 hypothesis PE",        None,                     True),
    ("crlongtrkdiry",       c("slc", "nuid", "crlongtrkdiry"),          "CR long-track dir. y",      None,                     False),
    ("vtx_x",               c("slc", "vertex", "x"),                    "Slice vertex x [cm]",       np.linspace(-200, 200, 41), False),
    ("vtx_y",               c("slc", "vertex", "y"),                    "Slice vertex y [cm]",       np.linspace(-200, 200, 41), False),
    ("vtx_z",               c("slc", "vertex", "z"),                    "Slice vertex z [cm]",       np.linspace(0, 500, 51),  False),
    ("n_trk",               c("n_trk"),                                 "Track-like PFPs per slice", np.arange(-0.5, 12.5, 1), False),
]
for name, col, xlabel, bins, logy in SLICE_VARS:
    plot("slc_" + name, col, xlabel, bins, logy)

## 6. Longest and second-longest track per slice
Ranked by length only; not a muon/pion assignment.

In [ ]:
TRACK_VARS = [
    ("len",                 ("trk", "len"),                       "Track length [cm]",             np.linspace(0, 400, 81), True),
    ("trackScore",          ("trackScore",),                      "Track score",                   np.linspace(0.5, 1, 51), True),
    ("theta_z",             ("theta_z",),                         "Angle to z [deg]",              np.linspace(0, 180, 37), False),
    ("dist_vtx",            ("dist_vtx",),                        "Start-to-slice-vertex [cm]",    np.linspace(0, 100, 51), True),
    ("rangeP_mu",           ("trk", "rangeP", "p_muon"),          "rangeP (μ hypothesis) [GeV/c]", np.linspace(0, 2, 51),   True),
    ("rangeP_pi",           ("trk", "rangeP", "p_pion"),          "rangeP (π hypothesis) [GeV/c]", np.linspace(0, 2, 51),   True),
    ("chi2_muon_pl2",       ("trk", "chi2pid", "I2", "chi2_muon"),   "χ²(μ), plane 2",   None, True),
    ("chi2_pion_pl2",       ("trk", "chi2pid", "I2", "chi2_pion"),   "χ²(π), plane 2",   None, True),
    ("chi2_proton_pl2",     ("trk", "chi2pid", "I2", "chi2_proton"), "χ²(p), plane 2",   None, True),
    ("pida_pl2",            ("trk", "chi2pid", "I2", "pida"),        "PIDA, plane 2",    None, True),
    ("chi2_mu_over_pi_pl2", ("chi2_mu_over_pi_pl2",),             "χ²(μ)/χ²(π), plane 2",          np.linspace(0, 5, 51),   True),
]
for level, tag in [("long1", "Longest"), ("long2", "2nd-longest")]:
    for name, sub, xlabel, bins, logy in TRACK_VARS:
        plot(f"{level}_{name}", c(level, *sub), f"{tag} track: {xlabel}", bins, logy)

## 7. Statistics-only sensitivity (one bin)

Two stages, as hnlpi0: write one pyhf-input dict per sample (hnlpi0's schema and file names), then read them back and compute the expected 95% CL CLs limit on mu (pyhf, asymptotics, background-only Asimov data), converted to |U|^2 = U * sqrt(mu) since the rate goes as |U|^4.
`mc_stat=True` uses the BNB and HNL MC statistical errors (`mcstat_err`); `False` is data statistics only.
Set `IF_SAVE = True` to write the dicts to `PYHF_DIR` (files are never overwritten: change `SUFFIX`).

Optimistic: HNL samples include the CORSIKA overlay, but no off-beam sample, no systematics, and no mu-pi pair / invariant-mass cut yet.

In [ ]:
IF_SAVE = False
PYHF_DIR, SUFFIX = "/exp/sbnd/data/users/sdas1/hnl/mu_pi/pyhf_inputs", "onebin_statonly"

# var=None: one bin holding every selected slice
pyhf_inputs = {"mcbnb": ana.pyhf_input_dict(bnb, config.TARGET_POT, scale=W)}
for m in MASSES:
    pyhf_inputs[m] = ana.pyhf_input_dict(hnl_sig[m], config.TARGET_POT, scale=W_HNL[m], weighted=True,
                                         m=HNL_INFO[m]["hnlM"], U=HNL_INFO[m]["simU"])
if IF_SAVE:
    print(ana.save_pyhf_inputs(PYHF_DIR, SUFFIX, pyhf_inputs["mcbnb"], {m: pyhf_inputs[m] for m in MASSES}))
    pyhf_inputs = ana.load_pyhf_inputs(PYHF_DIR, SUFFIX, MASSES)

sens = ana.sensitivity_table(pyhf_inputs)
display(sens.style.format(precision=3, formatter={c: "{:.3g}" for c in sens.columns if c.startswith("|U|")}))

## 8. Track cuts
`ana.TRACK_CUTS` follow `DEFAULT_CUTS`, in the analysis order: track quality, MIP tracks, no primary shower, MIP-pair opening angle, containment, cathode crossers, Michel-like PFPs. They cut on per-slice counts of **every** neutrino-daughter PFP, computed in `preprocess.pfp_selection` (thresholds in `preprocess.PFP_CUTS`; change them with `functools.partial(ana.preprocess_mcbnb, ...)`). Reco only. Not yet implemented: muon/pion separation, the exactly-one-pion veto, the high y-z region, the high-energy-proton BDT.

In [ ]:
display(ana.cutflow(F0, ana.ALL_CUTS, weights={"bnb": W, **{f"m{m}": W_HNL[m] for m in MASSES}}).round(3))